## Task: Intelligent Routing System

Необходимо разработать сервис для автоматической обработки и распределения обращений клиентов, поступивших в нерабочие часы. Система должна проанализировать массив данных (CSV), использовать ИИ для обогащения информации (классификация, суммаризация, гео-кодинг) и назначить каждое обращение на наиболее подходящего сотрудника, соблюдая жесткие бизнес-правила и балансировку нагрузки.

* **Tickets (Обращения):** GUID клиента, Пол, Дата рождения, Сегмент (Mass, VIP, Priority), Описание (текст обращения, свободный ввод), Вложения (указатель на файл), Страна, Область, Населенный пункт, Улица, Дом.
* **Managers (Менеджеры):** ФИО, Должность (Спец, Ведущий спец, Глав спец), Навыки (массив: VIP, ENG, KZ), Бизнес-единица (офис), Кол-во обращений в работе (текущая нагрузка).
* **Business Units (Офисы):** Офис, Адрес.


### AI-анализ текста (NLP Модуль)

Каждое обращение должно быть проанализировано с помощью ИИ (LLM) для извлечения следующих атрибутов:
* **Тип обращения (`тип_обращения`):** Одна из категорий: Жалоба, Смена данных, Консультация, Претензия, Неработоспособность приложения, Мошеннические действия, Спам.
* **Тональность (`тональность_обращения`):** Определение эмоционального фона (Позитивный, Нейтральный, Негативный).
* **Приоритетность (`приоритет_обращения`):** Оценка срочности обработки по шкале от 1 до 10.
* **Язык обращения (`язык_обращения`):** Определение языка (KZ, ENG, RU). Если язык не определен по умолчанию считается RU.
* **Summary (`ии_выжимка`, `ии_рекомендация`):** Краткая выжимка сути обращения (1-2 предложения) + рекомендация по дальнейшим действиям для менеджера (1-2 предложения).
* **Гео-нормализация (`latitude`, `longitude`):** Преобразование текстового адреса клиента в координаты (широта/долгота) для расчета близости к офису.


### Логика распределения (Business Rules)

Сервис должен назначить обращение на менеджера, пройдя через каскад фильтров:
* **Географический фильтр:** Поиск ближайшего к клиенту офиса. Исключение: Клиенты с неизвестным адресом или из-за рубежа распределяются поровну (50/50) между офисами Астаны и Алматы.
* **Фильтр компетенций (Hard Skills):**
    * **VIP/Priority:** Могут обрабатывать только менеджеры с навыком VIP.
    * **Смена данных:** Могут обрабатывать только менеджеры с должностью Глав спец.
    * **Язык:** Если обращение на KZ или ENG, у менеджера должен быть соответствующий навык.
* **Балансировка (Round Robin):**
    * Внутри выбранного офиса выбираются два подходящих по скиллам менеджера с наименьшей текущей нагрузкой.
    * Обращения распределяются между ними по системе Round Robin (поочередно).

In [ ]:
import pandas as pd
import numpy as np

business_df = pd.read_excel("/kaggle/input/freedom-intelligent-routing-engine-fire/business_units.xlsx")
managers_df = pd.read_excel("/kaggle/input/freedom-intelligent-routing-engine-fire/managers.xlsx")
tickets_df = pd.read_excel("/kaggle/input/freedom-intelligent-routing-engine-fire/tickets.xlsx")

business_df.columns = [col.replace(" ", "_").lower() for col in business_df.columns]
managers_df.columns = [col.replace(" ", "_").lower() for col in managers_df.columns]
tickets_df.columns = [col.replace(" ", "_").lower() for col in tickets_df.columns]

print(f"Business Units {business_df.shape}:", business_df.columns.tolist())
print(f"Managers {managers_df.shape}:", managers_df.columns.tolist())
print(f"Tickets {tickets_df.shape}:", tickets_df.columns.tolist())

In [ ]:
import logging
from datetime import datetime, timedelta

def almaty_time(*args):
    return (datetime.utcnow() + timedelta(hours=5)).timetuple()

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    handlers=[
        logging.FileHandler("app.log", encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logging.Formatter.converter = almaty_time
logger = logging.getLogger(__name__)

## 1. GeoCoding / GeoPositioning Problem 

~ Get GeoLocations ~ GeoCoding task -> `latitude`, `longitude` by "address" values ~ 2Gis API (best for Kazakhstan and CIS regions)

| Метрика | Данные |
|---------|----------------------------|
| Доля рынка в Казахстане | 73% (уверенное доминирование в РК, для сравнения: Google Maps — 24%, Яндекс Карты — 3%). |
| Аудитория в Казахстане | Стабильно около 10 млн пользователей в месяц (в пиковые месяцы превышает 10,16 млн). |
| Глобальная аудитория | Более 70 миллионов пользователей собственных продуктов. |
| Покрытие | 7 стран (Казахстан, Россия, Кыргызстан, Узбекистан, Азербайджан, Беларусь, ОАЭ), более 1462 городов. |
| Бизнес-база | Более 5 миллионов компаний в справочнике с актуальными данными. |

In [ ]:
# https://docs.2gis.com/api/search/geocoder/overview

import re
import time
import requests

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

API_KEY_2GIS = user_secrets.get_secret("API_KEY_2GIS")

def geocode_2gis_safe(address: str, max_parts: int = 6, max_length: int = 200):
    """
    Геокодирование через API 2GIS с жадным поиском (greedy fallback)
    """
    if pd.isna(address) or not isinstance(address, str):
        return pd.Series([None, None])
    if len(address) > max_length:
        address = address[:max_length]
    cleaned = re.sub(r'[«»"“”]', '', address)
    parts = [p.strip() for p in cleaned.split(',') if p.strip()]
    if len(parts) > max_parts:
        parts = parts[:max_parts]
    while len(parts) > 0:
        query = ", ".join(parts)
        url = "https://catalog.api.2gis.com/3.0/items/geocode"
        params = {
            "q": query,
            "fields": "items.point",
            "key": API_KEY_2GIS
        }
        try:
            response = requests.get(url, params=params, timeout=5)
            data = response.json()
            if data.get("meta", {}).get("code") == 200 and "result" in data:
                items = data["result"].get("items", [])
                if len(items) > 0:
                    point = items[0].get("point")
                    if point:
                        logger.info(f"OK: 2Gis -> {query} -> {point['lat']}, {point['lon']}")
                        return pd.Series([point["lat"], point["lon"]])
        except Exception as e:
            logger.error(f"ERROR '{query}': {e}")
            return pd.Series([None, None])
        logger.warning(f"NOT FOUND: {query}. Reducing...")
        parts.pop()
        time.sleep(0.2)
    return pd.Series([None, None])

mask = business_df.apply(
    lambda row: str(row['офис']).lower() in str(row['адрес']).lower(),
    axis=1
)
business_df["full_address"] = np.where(
    mask,
    business_df["адрес"],  # если город уже есть — не добавляем
    business_df["офис"] + ", " + business_df["адрес"]  # иначе добавляем
)
business_df[['latitude', 'longitude']] = (
    business_df["full_address"]
).apply(geocode_2gis_safe)

business_df.head()

In [ ]:
address_cols = ["страна", "область", "населённый_пункт", "улица", "дом"]

tickets_df[address_cols] = tickets_df[address_cols].fillna("").astype(str)

tickets_df[address_cols] = (
    tickets_df[address_cols]
    .fillna("")
    .astype(str)
    .apply(lambda col: col.str.strip())
)

def build_address(row):
    parts = []
    for col in address_cols:
        value = row[col].strip()
        if value and value.lower() not in [p.lower() for p in parts]:
            parts.append(value)
    full = ", ".join(parts)
    full = re.sub(r"\s+", " ", full)
    return full[:200]

tickets_df["full_address"] = tickets_df.apply(build_address, axis=1)

tickets_df[['latitude', 'longitude']] = (
    tickets_df["full_address"]
    .apply(geocode_2gis_safe)
)

In [ ]:
import folium

map_center = folium.Map(location=[48, 68], zoom_start=5, tiles='CartoDB positron')

for _, row in business_df.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=6,
        color='darkgreen',
        fill=True,
        fill_color='darkgreen',
        fill_opacity=0.8,
        popup='Business Office'
    ).add_to(follium_map)

for _, row in tickets_df.iterrows():
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=4,
        color='#FFCC80', # Light Orange hex
        fill=True,
        fill_color='#FFCC80',
        fill_opacity=0.6,
        popup='Ticket Location'
    ).add_to(follium_map)

follium_map

## 2. NLP Module - AI Analysis (JSON-answer)

~ NLP Module -> Get Values `тип_обращения`, `тональность_обращения`, `приоритет_обращения`, `язык_обращения`, `ии_выжимка`, `ии_рекомендация`

| Параметр | Значение / Описание |
|----------|-------------------|
| Архитектура | Mixture-of-Experts (MoE), 16 экспертов (17B активных параметров, 109B всего). |
| Мультимодальность | Нативная поддержка текста и изображений (до 5 изображений по 20 MB в одном запросе). |
| Окно контекста | 131 072 (128K) токенов — позволяет загружать длинные логи и переписки. |
| Языковая поддержка | 12 языков (включая английский, немецкий, французский, испанский и др.). |
| Скорость вывода (Groq) | ~750 токенов в секунду (tps) благодаря аппаратным ускорителям Groq. |
| Стоимость API (Groq) | Ввод: 0.11d. за 1 млн токенов. Вывод: 0.34d. за 1 млн токенов. |
| Ключевые функции | Tool Use, JSON Object Mode, JSON Schema Mode. |

In [ ]:
tickets_df[tickets_df["вложения"].notna()]

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

counts = tickets_df["сегмент_клиента"].value_counts()

plt.figure(figsize=(4, 4))
counts.plot(
    kind="pie",
    autopct="%1.1f%%",
    startangle=90,
    ylabel="" 
)

plt.title("Распределение сегментов клиентов")
plt.tight_layout()
plt.show()

# Mass < VIP < Priority

In [ ]:
import base64
import os

def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def get_client_age(birth_date_val):
    if not birth_date_val:
        return None
    try:
        if isinstance(birth_date_val, str):
            birth_date = datetime.strptime(birth_date_val, "%Y-%m-%d").date()
        else:
            birth_date = birth_date_val
        today = datetime.today().date()
        age = today.year - birth_date.year - ((today.month, today.day) < (birth_date.month, birth_date.day))
        return age
    except (ValueError, TypeError):
        return None

system_instruction = """<system_instruction>
  <role>
    Ты — ведущий аналитик системы интеллектуальной маршрутизации (Intelligent Routing System). 
    Твоя задача: глубокий NLP-анализ входящих обращений для их последующего распределения между менеджерами.
  </role>

  <task_description>
    Проанализируй предоставленные данные клиента и текст его обращения. Обогати информацию метаданными, 
    соблюдая строгую типизацию и логику бизнес-процессов компании.
  </task_description>

  <classification_rules>
    <category_mapping>
      Определи "тип_обращения" строго из списка:
      - Жалоба
      - Смена данных
      - Консультация
      - Претензия
      - Неработоспособность приложения
      - Мошеннические действия
      - Спам
    </category_mapping>

    <sentiment_analysis>
      Определи "тональность_обращения": [Позитивный, Нейтральный, Негативный].
    </sentiment_analysis>

    <priority_logic>
      Присвой "приоритет_обращения" от 1 до 10.
      Критерии повышения: Сегмент VIP/Priority, признаки мошенничества, агрессивная тональность.
    </priority_logic>

    <language_detection>
      Определи "язык_обращения": [KZ, ENG, RU]. Если текст смешанный или неочевидный — используй RU.
    </language_detection>

    <ai_summary>
      Краткая выжимка обращения (1-2 предложения).
    </ai_summary>

    <ai_recommendation>
      Рекомендация для менеджера (1-2 предложения).
    </ai_recommendation>
  </classification_rules>

  <output_format>
    Верни ответ ТОЛЬКО в формате JSON. Не добавляй лишних пояснений вне структуры.
    {
      "тип_обращения": string,
      "тональность_обращения": string,
      "приоритет_обращения": integer,
      "язык_обращения": string,
      "ии_выжимка": string,
      "ии_рекомендация": string
    }
  </output_format>
</system_instruction>"""

In [ ]:
!pip install -qU groq

In [ ]:
# https://console.groq.com/docs/model/meta-llama/llama-4-scout-17b-16e-instruct
# https://huggingface.co/meta-llama/Llama-4-Scout-17B-16E-Instruct

from groq import Groq

GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
client = Groq(api_key=GROQ_API_KEY)

def get_groq_llama_ai_analysis_json(query: str, base64_image: str = None):
    query_content = [
        {"type": "text", "text": query}
    ]
    
    if base64_image:
        query_content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}
        })
    
    completion = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[
            {
                "role": "system",
                "content": system_instruction
            },
            {
                "role": "user",
                "content": query_content
            }
        ],
        temperature=0.1,
        max_completion_tokens=1024,
        top_p=1,
        stream=False,
        response_format={"type": "json_object"}
    )
    return completion.choices[0].message.content

In [ ]:
import json

EXPECTED_KEYS = [
    "тип_обращения", 
    "тональность_обращения", 
    "приоритет_обращения", 
    "язык_обращения", 
    "ии_выжимка", 
    "ии_рекомендация"
]

def preprocess_llm_json(raw_output):
    if not raw_output or not isinstance(raw_output, str):
        return {}
    try:
        clean_content = re.sub(r'```(?:json)?\n?|```', '', raw_output).strip()
        match = re.search(r'(\{.*\})', clean_content, re.DOTALL)
        if match:
            clean_content = match.group(1)
        return json.loads(clean_content)
    except (json.JSONDecodeError, ValueError) as e:
        print(f"Extraction Error: {e} | Raw: {raw_output[:50]}...")
        return {}

def get_json_based_on_client_row(row):
    base64_image = None
    
    attachment_name = row.get("вложения")
    if pd.notna(attachment_name) and attachment_name:
        file_path = os.path.join("/kaggle/input/freedom-intelligent-routing-engine-fire/attachments", str(attachment_name))
        if os.path.exists(file_path) and str(file_path).lower().endswith(("png", "jpeg", "jpg", "heic")):
            base64_image = encode_image(file_path)
    
    client_info = {
        "пол": row.get("пол_клиента"),
        "возраст": get_client_age(row.get("дата_рождения")),
        "сегмент_клиента": row.get("сегмент_клиента"),
        "страна": row.get("страна")
    }
    
    query = (
        f"Информация по клиенту: <client_info>{str(client_info)}</client_info>\n"
        f"Текст обращения клиента: <client_text>{row.get('описание')}</client_text>\n"
        "\nВерни результаты анализа в формате JSON!"
    )
    
    if base64_image:
        query += "\nТакже клиент прикрепил изображение по текущему запросу."

    try:
        output_json = preprocess_llm_json(get_groq_llama_ai_analysis_json(query, base64_image))
        # print(output_json)
        output_json = dict(output_json)
    except Exception as e:
        print(f"Error processing row: {e}")
        output_json = {}

    return pd.Series({key: output_json.get(key) for key in EXPECTED_KEYS})

In [ ]:
tickets_df[EXPECTED_KEYS] = tickets_df.apply(get_json_based_on_client_row, axis=1)

In [ ]:
# business_df.to_csv("business_with_locations.csv", index=None)
# tickets_df.to_csv("tickets_with_locations_and_ai_data.csv", index=None)

## 3. Routing Business Logic (Round Robin Support)

1) geographical closest office to the client, if client is from foreign country or undefined address -> choose less loaded from Astana - Almaty;
2) vip/priority can be processed only by VIP managers;
3) task_type: data change can be processed only by main specialists;
4) language must match: russian is default for all;
5) in the current office ~ less loaded available manager with enough skill will be chosen;
6) Round Robin: queue-based routing -> Round-robin based routing is a simple circular scheduling algorithm that distributes incoming network traffic, server requests, or tasks equally across a group of resources in a rotating, sequential order.

In [ ]:
import pandas as pd

managers_df = pd.read_excel("/kaggle/input/freedom-intelligent-routing-engine-fire/managers.xlsx")
business_df = pd.read_csv("/kaggle/input/freedom-intelligent-routing-engine-fire/processed/business_with_locations.csv")
tickets_df = pd.read_csv("/kaggle/input/freedom-intelligent-routing-engine-fire/processed/tickets_with_locations_and_ai_data.csv")

managers_df.columns = [col.replace(" ", "_").lower() for col in managers_df.columns]
managers_df["офис"] = (
    managers_df["офис"]
    .astype(str)
    .str.lower()
    .str.strip()
)
business_df["офис"] = (
    business_df["офис"]
    .astype(str)
    .str.lower()
    .str.strip()
)

managers_df.head()

In [ ]:
managers_df["навыки"].unique().tolist()

In [ ]:
tickets_df.head()

In [ ]:
business_df.head()

In [ ]:
import re

city_to_region = {
    "актау": "мангистауская",
    "актобе": "актюбинская",
    "алматы": "алматы",
    "астана": "астана",
    "атырау": "атырауская",
    "караганда": "карагандинская",
    "кокшетау": "акмолинская",
    "костанай": "костанайская",
    "кызылорда": "кызылординская",
    "павлодар": "павлодарская",
    "петропавловск": "северо-казахстанская",
    "тараз": "жамбылская",
    "уральск": "западно-казахстанская",
    "усть-каменогорск": "восточно-казахстанская",
    "шымкент": "шымкент"
}

def normalize_text(text: str) -> str:
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r"[^\w\s-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text

business_df["office_norm"] = business_df["офис"].apply(normalize_text)

region_to_office = {
    v: k for k, v in city_to_region.items()
}

### 3.1. Определение assigned_office

1. Если lat/lon отсутствуют или координаты вне Казахстана → выбрать менее нагруженный из Astana / Almaty
2. Иначе → ближайший офис из business_df

In [ ]:
print(business_df["офис"].tolist())

In [ ]:
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

# официальный источник Natural Earth (110m resolution)
url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"

world = gpd.read_file(url)

kazakhstan = world.loc[
    world["ADMIN"] == "Kazakhstan", "geometry"
].values[0]

def is_in_kazakhstan(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return False
    point = Point(lon, lat)
    return bool(kazakhstan.contains(point))

In [ ]:
core_offices = ["астана", "алматы"] # для тикетов иностранцев и неизвестных локации

office_load = (
    managers_df
    .groupby("офис")["количество_обращений_в_работе"]
    .sum()
    .to_dict()
)

print(office_load)

In [ ]:
def choose_less_loaded_core_office():
    astana_load = office_load.get("астана", 0)
    almaty_load = office_load.get("алматы", 0)
    return "астана" if astana_load <= almaty_load else "алматы"

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)

    a = np.sin(dphi/2)**2 + \
        np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    
    return 2 * R * np.arcsin(np.sqrt(a))

In [ ]:
%%time

def assign_office(ticket_row, business_df):
    lat = ticket_row["latitude"]
    lon = ticket_row["longitude"]
    country = normalize_text(ticket_row["страна"])
    
    # если не Казахстан → core офис
    if len(country) > 0 and country != "казахстан":
        return choose_less_loaded_core_office()
    
    if pd.isna(lat) or pd.isna(lon):
        return choose_less_loaded_core_office()
    
    if not is_in_kazakhstan(lat, lon):
        return choose_less_loaded_core_office()
    
    region = normalize_text(ticket_row["область"])
    life_point = normalize_text(ticket_row["населённый_пункт"])
    
    # full-match по городу
    if life_point in city_to_region:
        return life_point
        
    # prefix-match по области
    for region_key in region_to_office.keys():
        if region.startswith(region_key):
            return region_to_office[region_key]
        
    # ближайший офис
    distances = business_df.apply(
        lambda x: haversine(lat, lon, x["latitude"], x["longitude"]),
        axis=1
    )
    
    return business_df.iloc[distances.idxmin()]["офис"]

tickets_df["assigned_office"] = tickets_df.apply(
    lambda row: assign_office(row, business_df),
    axis=1
)

tickets_df.head()

### 3.2. Формируем очереди по офисам

In [ ]:
office_queues = {
    office: df.copy()
    for office, df in tickets_df.groupby("assigned_office")
}

### 3.3. Фильтрация менеджеров по навыкам

| Условие     | Требование                     |
| ----------- | ------------------------------ |
| VIP, Priority         | Навык VIP                      |
| Data change | Должность = Главный специалист |
| Язык ENG    | Навык ENG                      |
| Язык KZ     | Навык KZ                       |
| RU          | доступен всем                  |

In [ ]:
def filter_managers(ticket, managers_office_df, office):
    
    df = managers_office_df.copy()
    final_office = office
    
    # VIP
    if ticket["сегмент_клиента"] in ("VIP", "Priority"):
        df = df[df["навыки"].str.contains("VIP", na=False)]
    
    # Data change
    if ticket["тип_обращения"] == "Data change":
        df = df[df["должность"] == "Главный специалист"]
    
    # Language
    lang = ticket["язык_обращения"]
    
    if lang == "ENG":
        df = df[df["навыки"].str.contains("ENG", na=False)]
    elif lang == "KZ":
        df = df[df["навыки"].str.contains("KZ", na=False)]
    # RU — без фильтра

    if df.empty and office not in ("алматы", "астана"):
        final_office = choose_less_loaded_core_office()
        managers_office = managers_df[
            managers_df["офис"] == final_office
        ].copy()
        return filter_managers(ticket, managers_office, final_office)
    
    return df, final_office

### 3.4. Round Robin внутри офиса

Для каждого офиса:

Сортируем тикеты:

* Сначала VIP

* Потом по приоритету (descending)

* Потом остальные

In [ ]:
def sort_tickets(df):
    return (
        df
        .assign(
            vip_flag=lambda x: x["сегмент_клиента"]
                .isin(["VIP", "Priority"])
                .astype(int)
        )
        .sort_values(
            by=["vip_flag", "приоритет_обращения"],
            ascending=[False, False]
        )
        .drop(columns="vip_flag")
    )

In [ ]:
%%time

from collections import defaultdict

assignments = []

for office, tickets in office_queues.items():
    
    tickets = sort_tickets(tickets)
    
    managers_office = managers_df[
        managers_df["офис"] == office
    ].copy()
    
    # сортируем менеджеров по текущей нагрузке
    managers_office = managers_office.sort_values(
        "количество_обращений_в_работе"
    )
    
    rr_index = 0
    
    for _, ticket in tickets.iterrows():
        
        available, final_office = filter_managers(
            ticket,
            managers_office,
            office
        )
        
        if available.empty:
            continue
        
        manager_list = available["фио"].tolist()
        
        manager = manager_list[rr_index % len(manager_list)]
        rr_index += 1
        
        assignments.append({
            "guid_клиента": ticket["guid_клиента"],
            "assigned_manager": manager,
            "assigned_office": final_office
        })
        
        # увеличиваем нагрузку
        managers_office.loc[
            managers_office["фио"] == manager,
            "количество_обращений_в_работе"
        ] += 1

In [ ]:
assignments_df = pd.DataFrame(assignments)

result_df = tickets_df.merge(
    assignments_df,
    on="guid_клиента",
    how="left",
    suffixes=("", "_new")
)

result_df["assigned_office"] = (
    result_df["assigned_office_new"]
    .combine_first(result_df["assigned_office"])
)

result_df = result_df.drop(columns=["assigned_office_new"])

result_df.head()

In [ ]:
import folium
from folium.plugins import AntPath

m = folium.Map(location=[48, 68], zoom_start=5)

for _, office in business_df.iterrows():
    folium.Marker(
        location=[office["latitude"], office["longitude"]],
        popup=f'Офис: {office["офис"]}',
        icon=folium.Icon(color="green", icon="building")
    ).add_to(m)

for _, ticket in tickets_df.iterrows():
    
    # Координаты клиента
    client_lat = ticket["latitude"]
    client_lon = ticket["longitude"]
    
    if pd.isna(client_lat) or pd.isna(client_lon):
        continue
    
    # Ищем офис
    assigned = ticket["assigned_office"]
    office_row = business_df[
        business_df["офис"].str.lower() == str(assigned).lower()
    ]
    
    if office_row.empty:
        continue
    
    office_lat = office_row.iloc[0]["latitude"]
    office_lon = office_row.iloc[0]["longitude"]
    
    # Клиентская точка
    folium.CircleMarker(
        location=[client_lat, client_lon],
        radius=4,
        popup=f'Клиент: {ticket["guid_клиента"]}<br>Офис: {assigned}',
        color="blue",
        fill=True
    ).add_to(m)
    
    # Линия (анимированная)
    AntPath(
        locations=[[client_lat, client_lon], [office_lat, office_lon]],
        delay=800,
        weight=2
    ).add_to(m)

m

In [ ]:
# Средняя нагрузка менеджеров по офисам
manager_counts = result_df.groupby("assigned_office")["assigned_manager"].nunique()

load_stats = (
    result_df
    .groupby("assigned_office")["assigned_manager"]
    .value_counts()
    .groupby(level=0)
    .agg(["min", "max", "mean"])
)

load_stats["num_managers"] = manager_counts

load_stats = load_stats.sort_values(by="num_managers", ascending=False)

styled_table = load_stats.style.background_gradient(
    cmap="Greens",
    axis=None  # по всей таблице
).set_caption("Статистика нагрузки менеджеров по офисам")

styled_table

In [ ]:
# Среднее расстояние тикетов до офисов
result_df["distance_to_office"] = result_df.apply(
    lambda row: haversine(
        row["latitude"],
        row["longitude"],
        business_df.loc[business_df["офис"] == row["assigned_office"], "latitude"].values[0],
        business_df.loc[business_df["офис"] == row["assigned_office"], "longitude"].values[0]
    ),
    axis=1
)

avg_distance = result_df.groupby("assigned_office")["distance_to_office"].mean().sort_values(ascending=False)
avg_distance_df = avg_distance.reset_index()
avg_distance_df.columns = ["Офис", "Среднее расстояние, км"]

styled_distance = (
    avg_distance_df.style
    .background_gradient(cmap="Greens", axis=None)
    .format({"Среднее расстояние, км": "{:.2f}"})
    .set_caption("Среднее расстояние тикетов до офисов (км)")
)

styled_distance